**TASK 1:- Set up the LLM API connection**

Install Required Libraries

In [41]:
# Required libraries install kar rahe hain
!pip install jsonschema requests

Import Required Libraries

In [42]:
# Required libraries import kar rahe hain
import os
import re
import json
import joblib
import requests
import pandas as pd

# JSON schema validation ke liye required libraries import kar rahe hain
from jsonschema import validate
from jsonschema.exceptions import ValidationError

 basic information define kar rahe hain

In [43]:

TRACK = "Track C - Model Prediction Explanation Pipeline"

MODEL_NAME = "openai/gpt-4o-mini"

API_URL = "https://openrouter.ai/api/v1/chat/completions"

print("Assignment Started Successfully")

print(TRACK)

Assignment Started Successfully
Track C - Model Prediction Explanation Pipeline


API Key

In [40]:
# API key ko environment variable me securely store kar rahe hain
os.environ["LLM_API_KEY"] = input("Enter your OpenRouter API Key: ")

# Environment variable se API key read kar rahe hain
api_key = os.getenv("LLM_API_KEY")

# Check kar rahe hain ki API key successfully load hui ya nahi
if api_key:

    print("API Key Loaded Successfully.")

else:

    print("API Key Not Found.")

Enter your OpenRouter API Key: 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
API Key Loaded Successfully.


Load Best Model

In [44]:
# Part 3 me save ki hui best model load kar rahe hain
best_model = joblib.load("best_model.pkl")

# Success message display kar rahe hain
print("Best Model Loaded Successfully.")

Best Model Loaded Successfully.


Display Loaded Model

In [45]:
# Loaded machine learning model display kar rahe hain
print(best_model)

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(max_depth=10, min_samples_leaf=5,
                                        n_estimators=200, random_state=42))])


Task 1 — Reusable LLM Function

Step 1 — Define Function

In [46]:
# LLM API ko reusable tarike se call karne ke liye function bana rahe hain
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    # Request headers define kar rahe hain
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "X-Title": "Machine Learning Assignment"
    }

    # API payload create kar rahe hain
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    # API request send kar rahe hain
    response = requests.post(
        API_URL,
        headers=headers,
        json=payload
    )

    # Response status check kar rahe hain
    if response.status_code != 200:
        print("Status Code:", response.status_code)
        print(response.text)
        return None

    # JSON response return kar rahe hain
    result = response.json()

    # Final response return kar rahe hain
    return result["choices"][0]["message"]["content"]

STEP 2: Test the Reusable Function

In [47]:
# Reusable function ko test karne ke liye system prompt define kar rahe hain
system_prompt = "You are a helpful AI assistant. Reply only with the final answer."

# Test ke liye user prompt define kar rahe hain
user_prompt = "Reply with only the word: hello"

# Reusable function ko call kar rahe hain
response = call_llm(
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    temperature=0.0,
    max_tokens=20
)

# LLM ka response display kar rahe hain
print(response)

hello


Create System Prompt

In [48]:
# LLM ke behaviour ko control karne ke liye system prompt define kar rahe hain
SYSTEM_PROMPT = """
You are an expert Machine Learning Prediction Explanation Assistant.

Your task is to explain machine learning predictions clearly and professionally.

Rules:

1. Explain the prediction in simple language.
2. Mention the predicted class.
3. Mention the confidence score.
4. Explain the most influential features.
5. Do not generate false information.
6. Return only the requested output.
"""

Create Sample Test Data

In [49]:
# Model prediction ko test karne ke liye sample input data create kar rahe hain
sample_input = pd.DataFrame({

    "age": [22],
    "platforms_used_count": [4],
    "daily_screen_hours": [8.5],
    "daily_notifications": [180],
    "minutes_to_first_check_after_waking": [4],
    "avg_sleep_hours": [5.5],
    "anxiety_score_0to27": [18],
    "depression_score_0to27": [12],
    "attention_score_0to10": [5],
    "productivity_score_0to10": [4],
    "social_isolation_score_0to10": [7],
    "fomo_1to10": [9]

})

# Sample input display kar rahe hain
sample_input

,age,platforms_used_count,daily_screen_hours,daily_notifications,minutes_to_first_check_after_waking,avg_sleep_hours,anxiety_score_0to27,depression_score_0to27,attention_score_0to10,productivity_score_0to10,social_isolation_score_0to10,fomo_1to10
0,22,4,8.5,180,4,5.5,18,12,5,4,7,9


In [50]:
# Loaded model ke expected feature names dekh rahe hain
print(best_model.feature_names_in_)

['age' 'platforms_used_count' 'daily_screen_hours' 'daily_notifications'
 'night_time_use' 'minutes_to_first_check_after_waking' 'avg_sleep_hours'
 'anxiety_score_0to27' 'life_satisfaction_1to10' 'loneliness_1to10'
 'self_esteem_1to10' 'fomo_1to10' 'social_comparison_1to10'
 'physical_activity_days_per_week' 'wellbeing_band' 'gender_Male'
 'gender_Non-binary' 'gender_Prefer not to say'
 'occupation_Part-time employed' 'occupation_Retired'
 'occupation_Self-employed' 'occupation_Student' 'occupation_Unemployed'
 'region_Asia' 'region_Europe' 'region_Latin America'
 'region_North America' 'region_Oceania' 'most_used_platform_Instagram'
 'most_used_platform_LinkedIn' 'most_used_platform_Reddit'
 'most_used_platform_Snapchat' 'most_used_platform_TikTok'
 'most_used_platform_X/Twitter' 'most_used_platform_YouTube'
 'primary_purpose_Content creation' 'primary_purpose_Entertainment'
 'primary_purpose_News/information' 'primary_purpose_Passing time/boredom'
 'primary_purpose_Work/career' 'uses

Load One Sample from Dataset

In [51]:
# Part 3 me save ki hui encoded feature dataset load kar rahe hain
X = pd.read_csv("encoded_features.csv")

# Dataset ka size check kar rahe hain
print("Dataset Shape :", X.shape)

# First 5 rows display kar rahe hain
X.head()

Dataset Shape : (7000, 45)


,age,platforms_used_count,daily_screen_hours,daily_notifications,night_time_use,minutes_to_first_check_after_waking,avg_sleep_hours,anxiety_score_0to27,life_satisfaction_1to10,loneliness_1to10,...,primary_purpose_Content creation,primary_purpose_Entertainment,primary_purpose_News/information,primary_purpose_Passing time/boredom,primary_purpose_Work/career,uses_screen_time_limits_Yes,"attempted_digital_detox_Yes, failed","attempted_digital_detox_Yes, succeeded",seeks_mental_health_support_No,seeks_mental_health_support_Yes
0,33,8,3.2,172,0,25,8.7,17,9,6,...,False,True,False,False,False,False,False,False,True,False
1,23,8,4.5,38,2,26,7.3,18,7,5,...,False,True,False,False,False,False,True,False,False,True
2,56,1,5.3,74,3,12,7.0,12,8,7,...,False,True,False,False,False,False,False,False,True,False
3,13,4,3.4,49,1,21,7.0,13,10,6,...,False,False,True,False,False,False,True,False,False,True
4,36,5,5.8,227,0,19,6.7,15,4,6,...,False,False,False,False,True,True,False,True,True,False


Prediction ke liye sample row lenge.

In [52]:
# Prediction ke liye first sample select kar rahe hain
sample_input = X.iloc[[0]]

# Selected sample display kar rahe hain
sample_input

,age,platforms_used_count,daily_screen_hours,daily_notifications,night_time_use,minutes_to_first_check_after_waking,avg_sleep_hours,anxiety_score_0to27,life_satisfaction_1to10,loneliness_1to10,...,primary_purpose_Content creation,primary_purpose_Entertainment,primary_purpose_News/information,primary_purpose_Passing time/boredom,primary_purpose_Work/career,uses_screen_time_limits_Yes,"attempted_digital_detox_Yes, failed","attempted_digital_detox_Yes, succeeded",seeks_mental_health_support_No,seeks_mental_health_support_Yes
0,33,8,3.2,172,0,25,8.7,17,9,6,...,False,True,False,False,False,False,False,False,True,False


Prediction karenge.

In [53]:
# Selected sample par prediction generate kar rahe hain
prediction = best_model.predict(sample_input)

# Prediction probability calculate kar rahe hain
prediction_probability = best_model.predict_proba(sample_input)

# Prediction results display kar rahe hain
print("Predicted Class :", prediction[0])

print("Prediction Probability :")

print(prediction_probability)

Predicted Class : 0
Prediction Probability :
[[0.64443789 0.35556211]]


Convert Prediction into Human Readable Format

In [54]:
# Prediction aur probability ko readable format me convert kar rahe hain
predicted_class = int(prediction[0])

confidence_score = round(
    prediction_probability[0][predicted_class] * 100,
    2
)

# Prediction summary display kar rahe hain
print("Prediction Summary")
print("-" * 40)

print(f"Predicted Class : {predicted_class}")

print(f"Confidence Score : {confidence_score}%")

Prediction Summary
----------------------------------------
Predicted Class : 0
Confidence Score : 64.44%


Create User Prompt for LLM

In [55]:
# Prediction ko explain karne ke liye user prompt prepare kar rahe hain
user_prompt = f"""
The machine learning model predicted the following:

Predicted Class: {predicted_class}

Confidence Score: {confidence_score}%

The input features are:

{sample_input.to_string(index=False)}

Please explain the prediction in simple language.

Return your response ONLY as valid JSON using this structure:

{{
    "prediction": "",
    "confidence": "",
    "summary": "",
    "important_features": [],
    "recommendation": ""
}}

Do not return markdown.

Do not return code fences.

Return JSON only.
"""

Call the LLM

In [57]:
# Prediction explanation generate karne ke liye LLM ko call kar rahe hain
llm_response = call_llm(

    system_prompt=SYSTEM_PROMPT,

    user_prompt=user_prompt,

    temperature=0.0,

    max_tokens=500

)

# Raw LLM response display kar rahe hain
print(llm_response)

{
    "prediction": "The model predicts that the individual belongs to class 0.",
    "confidence": "64.44%",
    "summary": "This means the model is somewhat confident (about 64%) in its prediction that this person fits into the category represented by class 0. The prediction is based on various factors related to their lifestyle and mental health.",
    "important_features": [
        "daily_notifications: 172",
        "anxiety_score_0to27: 17",
        "avg_sleep_hours: 8.7",
        "daily_screen_hours: 3.2",
        "gender_Male: True"
    ],
    "recommendation": "Consider exploring ways to reduce daily notifications and manage anxiety, as these factors seem to influence the prediction significantly."
}


Parse JSON Response

In [58]:
# LLM response ko Python dictionary me convert kar rahe hain
response_json = json.loads(llm_response)

# Parsed JSON display kar rahe hain
response_json

{'prediction': 'The model predicts that the individual belongs to class 0.',
 'confidence': '64.44%',
 'summary': 'This means the model is somewhat confident (about 64%) in its prediction that this person fits into the category represented by class 0. The prediction is based on various factors related to their lifestyle and mental health.',
 'important_features': ['daily_notifications: 172',
  'anxiety_score_0to27: 17',
  'avg_sleep_hours: 8.7',
  'daily_screen_hours: 3.2',
  'gender_Male: True'],
 'recommendation': 'Consider exploring ways to reduce daily notifications and manage anxiety, as these factors seem to influence the prediction significantly.'}

Define JSON Schema

In [59]:
# LLM response validate karne ke liye JSON schema define kar rahe hain
response_schema = {

    "type": "object",

    "properties": {

        "prediction": {
            "type": "string"
        },

        "confidence": {
            "type": "string"
        },

        "summary": {
            "type": "string"
        },

        "important_features": {

            "type": "array",

            "items": {
                "type": "string"
            }

        },

        "recommendation": {
            "type": "string"
        }

    },

    "required": [

        "prediction",

        "confidence",

        "summary",

        "important_features",

        "recommendation"

    ]

}

Validate JSON

In [77]:
# Generated JSON ko validate kar rahe hain aur failure par fallback return kar rahe hain
try:

    validate(
        instance=response_json,
        schema=response_schema
    )

    print("JSON Validation Successful")

except ValidationError as error:

    print("Validation Failed")

    print(error)

    response_json = fallback_response

    print("\nFallback Response:")

    print(response_json)

JSON Validation Successful


Create PII Guardrail Function

Ye function email aur phone number ko mask karega.

In [69]:
# Sensitive personal information ko detect aur mask karne ke liye function bana rahe hain
def mask_pii(text):

    # Email address ko mask kar rahe hain
    text = re.sub(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        '[EMAIL REDACTED]',
        text
    )

    # Mobile number ko mask kar rahe hain
    text = re.sub(
        r'\b\d{10}\b',
        '[PHONE REDACTED]',
        text
    )

    return text

    # User input me PII detect karne ke liye function bana rahe hain
def has_pii(text):

    # Email pattern define kar rahe hain
    email_pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'

    # Phone number pattern define kar rahe hain
    phone_pattern = r'\b\d{10}\b'

    # Agar email ya phone number milta hai to True return kar rahe hain
    if re.search(email_pattern, text):
        return True

    if re.search(phone_pattern, text):
        return True

    # Agar koi PII nahi mila to False return kar rahe hain
    return False

Create Safe LLM Wrapper

In [70]:
# PII check karne ke baad hi LLM ko call karne ke liye wrapper function bana rahe hain
def safe_call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    # User prompt me PII check kar rahe hain
    if has_pii(user_prompt):

        print("PII detected. LLM request blocked.")

        return None

    # Agar PII nahi hai to LLM call kar rahe hain
    return call_llm(
        system_prompt,
        user_prompt,
        temperature,
        max_tokens
    )

Test the PII Guardrail

In [74]:
# PII masking ko test karne ke liye sample text define kar rahe hain
sample_text = """
Name: Rahul

Email: rahul123@gmail.com

Phone: 9876543210
"""

# Original text display kar rahe hain
print("Original Text")

print(sample_text)

print("-" * 50)

# Masked text display kar rahe hain
print("Masked Text")

print(mask_pii(sample_text))

Original Text

Name: Rahul

Email: rahul123@gmail.com

Phone: 9876543210

--------------------------------------------------
Masked Text

Name: Rahul

Email: [EMAIL REDACTED]

Phone: [PHONE REDACTED]



Test 1 (Input Contains Email → BLOCK)

In [72]:
# Email wala input create karke PII guardrail test kar rahe hain
pii_input = """
Customer Name: Rahul Sharma
Email: rahul123@gmail.com

Please explain this prediction.
"""

# Guardrail test run kar rahe hain
response = safe_call_llm(

    SYSTEM_PROMPT,

    pii_input,

    temperature=0.0,

    max_tokens=100

)

# Response display kar rahe hain
print(response)

PII detected. LLM request blocked.
None


Test 2 (Clean Input → ALLOW)

In [73]:
# PII ke bina clean input create kar rahe hain
clean_input = """
Predicted Class: 0

Confidence Score: 64.44%

Please explain this prediction.
"""

# Guardrail test run kar rahe hain
response = safe_call_llm(

    SYSTEM_PROMPT,

    clean_input,

    temperature=0.0,

    max_tokens=150

)

# LLM response display kar rahe hain
print(response)

The prediction indicates that the model classifies the input data into class 0. This means that based on the features provided, the model believes that the most likely category for this instance is class 0.

The confidence score of 64.44% suggests that the model is somewhat confident in this prediction, but it is not overwhelmingly certain. A score above 60% indicates a reasonable level of confidence, but there is still a significant chance (35.56%) that the prediction could be incorrect.

The most influential features that contributed to this prediction likely include specific characteristics or attributes of the input data that the model identified as important indicators for classifying it as class 0. These features could be numerical values, categorical data, or other relevant information that


Create Fallback Dictionary

In [75]:
# Validation fail hone par use karne ke liye fallback response define kar rahe hain
fallback_response = {

    "prediction": None,

    "confidence": None,

    "summary": None,

    "important_features": None,

    "recommendation": None

}

Generate Response at Temperature = 0.0

In [63]:
# Temperature 0.0 par LLM response generate kar rahe hain
response_temp_0 = call_llm(

    system_prompt=SYSTEM_PROMPT,

    user_prompt=user_prompt,

    temperature=0.0,

    max_tokens=500

)

# Temperature 0.0 ka response display kar rahe hain
print("Temperature = 0.0")

print("-" * 50)

print(response_temp_0)

Temperature = 0.0
--------------------------------------------------
{
    "prediction": "The model predicts that the individual belongs to class 0.",
    "confidence": "64.44%",
    "summary": "This means the model is somewhat confident in its prediction, indicating that there are characteristics in the data that align with class 0, but there is still a significant level of uncertainty.",
    "important_features": [
        "age: 33",
        "platforms_used_count: 8",
        "daily_screen_hours: 3.2",
        "daily_notifications: 172",
        "anxiety_score_0to27: 17",
        "life_satisfaction_1to10: 9",
        "loneliness_1to10: 6",
        "self_esteem_1to10: 4"
    ],
    "recommendation": "Consider exploring ways to reduce daily notifications and improve self-esteem, as these factors may influence the prediction."
}


Generate Response at Temperature = 0.7

In [64]:
# Temperature 0.7 par LLM response generate kar rahe hain
response_temp_07 = call_llm(

    system_prompt=SYSTEM_PROMPT,

    user_prompt=user_prompt,

    temperature=0.7,

    max_tokens=500

)

# Temperature 0.7 ka response display kar rahe hain
print("Temperature = 0.7")

print("-" * 50)

print(response_temp_07)

Temperature = 0.7
--------------------------------------------------
{
    "prediction": "The model predicts the class as 0.",
    "confidence": "64.44%",
    "summary": "This means that the model is moderately confident in its prediction, suggesting that the individual may belong to a group characterized by certain behaviors or traits associated with class 0.",
    "important_features": [
        "Age: 33",
        "Platforms Used Count: 8",
        "Daily Screen Hours: 3.2",
        "Daily Notifications: 172",
        "Average Sleep Hours: 8.7",
        "Anxiety Score: 17",
        "Life Satisfaction: 9",
        "Loneliness: 6",
        "Social Comparison: 4"
    ],
    "recommendation": "Consider focusing on reducing daily screen hours and managing notifications, as these aspects might improve overall wellbeing."
}


Compare Both Responses

In [65]:
# Dono temperature outputs ko compare kar rahe hain
print("Temperature Comparison")
print("=" * 60)

print("\nResponse at Temperature = 0.0")
print("-" * 60)
print(response_temp_0)

print("\nResponse at Temperature = 0.7")
print("-" * 60)
print(response_temp_07)

Temperature Comparison

Response at Temperature = 0.0
------------------------------------------------------------
{
    "prediction": "The model predicts that the individual belongs to class 0.",
    "confidence": "64.44%",
    "summary": "This means the model is somewhat confident in its prediction, indicating that there are characteristics in the data that align with class 0, but there is still a significant level of uncertainty.",
    "important_features": [
        "age: 33",
        "platforms_used_count: 8",
        "daily_screen_hours: 3.2",
        "daily_notifications: 172",
        "anxiety_score_0to27: 17",
        "life_satisfaction_1to10: 9",
        "loneliness_1to10: 6",
        "self_esteem_1to10: 4"
    ],
    "recommendation": "Consider exploring ways to reduce daily notifications and improve self-esteem, as these factors may influence the prediction."
}

Response at Temperature = 0.7
------------------------------------------------------------
{
    "prediction": "T

Select Three Different Test Inputs

In [66]:
# End-to-end testing ke liye teen alag-alag samples select kar rahe hain
test_inputs = X.iloc[[0, 10, 25]]

# Selected samples display kar rahe hain
test_inputs

,age,platforms_used_count,daily_screen_hours,daily_notifications,night_time_use,minutes_to_first_check_after_waking,avg_sleep_hours,anxiety_score_0to27,life_satisfaction_1to10,loneliness_1to10,...,primary_purpose_Content creation,primary_purpose_Entertainment,primary_purpose_News/information,primary_purpose_Passing time/boredom,primary_purpose_Work/career,uses_screen_time_limits_Yes,"attempted_digital_detox_Yes, failed","attempted_digital_detox_Yes, succeeded",seeks_mental_health_support_No,seeks_mental_health_support_Yes
0,33,8,3.2,172,0,25,8.7,17,9,6,...,False,True,False,False,False,False,False,False,True,False
10,28,3,1.7,38,2,55,8.1,7,6,1,...,False,False,False,False,True,True,True,False,False,True
25,29,3,6.5,35,2,28,6.0,14,8,6,...,False,False,False,False,False,False,False,False,True,False


Run Complete Pipeline on All Three Inputs

In [67]:
# Teen inputs par complete pipeline run karne ke liye empty list bana rahe hain
results = []

# Har sample par prediction aur LLM explanation generate kar rahe hain
for index, sample in test_inputs.iterrows():

    # Sample ko DataFrame me convert kar rahe hain
    sample_df = sample.to_frame().T

    # Prediction generate kar rahe hain
    prediction = best_model.predict(sample_df)[0]

    # Prediction probability calculate kar rahe hain
    probability = best_model.predict_proba(sample_df)[0][prediction]

    # User prompt prepare kar rahe hain
    prompt = f"""
Predicted Class: {prediction}

Confidence Score: {round(probability*100,2)}%

Input Features:

{sample_df.to_string(index=False)}

Return ONLY valid JSON using this format:

{{
    "prediction":"",
    "confidence":"",
    "summary":"",
    "important_features":[],
    "recommendation":""
}}
"""

    # LLM response generate kar rahe hain
    llm_output = call_llm(
        SYSTEM_PROMPT,
        prompt,
        temperature=0.0,
        max_tokens=500
    )

    # JSON validation check kar rahe hain
    try:

        parsed = json.loads(llm_output)

        validate(
            instance=parsed,
            schema=response_schema
        )

        validation = "PASS"

    except:

        validation = "FAIL"

    # Guardrail apply kar rahe hain
    guarded_output = mask_pii(llm_output)

    if guarded_output == llm_output:

        guardrail = "PASS"

    else:

        guardrail = "BLOCK"

    # Result list me save kar rahe hain
    results.append({

        "Input": index,

        "LLM Output": llm_output,

        "Valid JSON": validation,

        "Pass/Block": guardrail

    })

Display Results

In [68]:
# Final results ko DataFrame me convert kar rahe hain
results_df = pd.DataFrame(results)

# Results display kar rahe hain
results_df

,Input,LLM Output,Valid JSON,Pass/Block
0,0,"{\n ""prediction"":""0"",\n ""confidence"":""64...",PASS,PASS
1,10,"{\n ""prediction"":""0"",\n ""confidence"":""50...",PASS,PASS
2,25,"{\n ""prediction"":""0"",\n ""confidence"":""74...",PASS,PASS
